# 1. Libraries and setup

We will build a **Conditional Variational Autoencoder (cVAE)** on MNIST.

Each training example is now a pair:

$$
(x_i, c_i),
$$

where:

- $x_i$ is a handwritten digit image,
- $c_i \in \{0,\dots,9\}$ is its digit label.

Later, we will give $c_i$ to both the encoder and decoder:

$$
q_\phi(z \mid x_i, c_i),
\qquad
p_\theta(x_i \mid z, c_i).
$$

For generation, we will choose a desired digit label $c$, sample a random latent vector

$$
z \sim \mathcal{N}(0, I),
$$

and decode:

$$
(z,c) \longrightarrow x_{\mathrm{new}}.
$$

In [ ]:
# ── 1. Libraries and setup ────────────────────────────────────────────────────

import random
from pathlib import Path


import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms


# ── Reproducibility ───────────────────────────────────────────────────────────

SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Set random seeds for reproducible notebook results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ── Device ────────────────────────────────────────────────────────────────────
# Uses Apple Silicon GPU on a MacBook when MPS is available.

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")


# ── Plot settings and output directory ────────────────────────────────────────

FIG_DIR = Path("figures_cvae_mnist")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# 2. Load paired MNIST data

For a Conditional VAE, each training example is a pair:

$$
(x_i, c_i),
$$

where:

- $x_i$ is a handwritten MNIST digit image,
- $c_i \in \{0,\ldots,9\}$ is the digit label associated with that image.

For example:

$$
(x_i, c_i)
=
(\text{image of a handwritten 7},\, 7).
$$

At this stage, $c_i$ is stored as an integer label. Later, before passing it into the encoder and decoder, we will represent it as a numerical condition vector.

In [ ]:
# ── 2. Load paired MNIST data ─────────────────────────────────────────────────

from pathlib import Path

import matplotlib.pyplot as plt
from torchvision import datasets, transforms


# ── Dataset location ──────────────────────────────────────────────────────────

DATA_DIR = Path("data")


# ── MNIST transform ───────────────────────────────────────────────────────────
# Convert each 28 × 28 image into a PyTorch tensor with pixel values in [0, 1].

transform = transforms.ToTensor()


# ── Load MNIST ────────────────────────────────────────────────────────────────
# Each dataset item is already a pair:
#
#     (x_i, c_i)
#
# where:
#     x_i : digit image, shape (1, 28, 28)
#     c_i : digit label, integer in {0, ..., 9}

train_dataset_full = datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform,
)


# ── Inspect one paired example ─────────────────────────────────────────────────

print(f"Full training dataset: {len(train_dataset_full):,} paired examples")
print(f"Test dataset:          {len(test_dataset):,} paired examples")

x_example, c_example = train_dataset_full[0]

print(f"\nExample image shape: {tuple(x_example.shape)}")
print(f"Example condition c_i: {c_example}")
print(
    f"Pixel-value range: "
    f"[{x_example.min().item():.1f}, {x_example.max().item():.1f}]"
)


# ── Visualize paired examples ──────────────────────────────────────────────────

N_SHOW = 12

fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))

for ax, idx in zip(axes.flat, range(N_SHOW)):
    x_i, c_i = train_dataset_full[idx]

    ax.imshow(x_i.squeeze(0), cmap="gray")
    ax.set_title(rf"$c_i = {c_i}$")
    ax.axis("off")

fig.suptitle("MNIST paired data: image $x_i$ and condition $c_i$", y=1.03)
plt.tight_layout()
plt.show()